# 4. Measurements and clinical-style change points

1. Gain: the median of the nine selected bipoles at the intact reference placement is set to
   1.44 mV (imposed normalisation, not validation).
2. Per condition and noise realisation (0 = clean, then 3 x 5 uV Gaussian): unipole E5 unfiltered;
   bipole E5-E6|E9 and omnipoles E5-E6-E9, E9-E10-E5 band-passed 30-450 Hz (causal); peak-to-peak in
   the window. Selected weights are saved and applied to the clean signals.
3. Filter-history and profile checks (stable far plateau, lesion transition).
4. Change points: decay-to-plateau fits as in the clinical analysis, one slab, profile least squares.
   Results are eligible for interpretation only if every prerequisite passed and the matched intact
   control has no supported change point.

In [ ]:
import os, sys
from pathlib import Path
REPOSITORY = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPOSITORY))

# ---- settings: edit here, or set EFOV_CONFIG / EFOV_OUT / EFOV_NP ----
CONFIG = os.environ.get("EFOV_CONFIG", str(REPOSITORY / "config" / "final_model.json"))
OUT    = os.environ.get("EFOV_OUT",    str(REPOSITORY / "runs" / "v3_reference"))
NP     = int(os.environ.get("EFOV_NP", "24"))      # MPI processes for openCARP
# openCARP's own MPI launcher (efov-runner image); needed when NP > 1
os.environ.setdefault("MPIEXEC_BIN", "/usr/local/lib/opencarp/lib/petsc/bin/mpiexec")

import json
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display
from efov.config import loadConfig
from efov.pipeline import Run

config = loadConfig(CONFIG)
run = Run(config, OUT, processCount=NP)
print(f"{config['modelName']} -> {run.out}")
run.status()

In [ ]:
measures = run.runStage("measurements")
measures["calibration"], measures["checks"]

In [ ]:
pd.read_csv(run.out / "measurements" / "profileChecks.csv")

In [ ]:
changes = run.runStage("changePoints")
print("prerequisites:", changes["prerequisites"])
pd.DataFrame(changes["primary"])

In [ ]:
from efov.report import profileFigure
display(Image(str(profileFigure(run, run.folder("figures")))))

### Optional: Bayesian change points (R, mcp + JAGS; available in the efov-runner image)

In [ ]:
# import subprocess
# subprocess.run(["Rscript", str(REPOSITORY / "R" / "bayesian_change_points.R"), str(run.out)], check=True)
# pd.read_csv(run.out / "changePoints" / "bayesian" / "bayesianChangePoints.csv")